In [6]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Com autónoma": "Andalucía",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Provincia':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Provincia:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Provincia                  | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------
Sevilla                        |  Sevilla                   |      684234 |     4896.55 |    11 |   25.1 |   12.9 |      29 |    34.2
Lora del Río                   |  Sevilla                   |       18122 |        61.7 |    38 |   25.0 |   12.3 |      31 |    34.2
Lebrija                        |  Sevilla                   |       27616 |       73.05 |    36 |   24.1 |   12.1 |      28 |    35.9
Arcos de la Frontera           |  Cádiz                     |       30902 |       58.81 |   185 |   23.8 |   11.9 |      28 |    36.5
Trebujena                      |  Cádiz                     |        7000 |      102.45 |    69 |   23.8 |   12.6 |      27 |    36.5
Espera                         |  Cádiz                 

In [4]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Provincia": "Sevilla",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Provincia':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Provincia:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Provincia                  | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------
Sevilla                        |  Sevilla                   |      684234 |     4896.55 |    11 |   25.1 |   12.9 |      29 |    34.2
Lora del Río                   |  Sevilla                   |       18122 |        61.7 |    38 |   25.0 |   12.3 |      31 |    34.2
Lebrija                        |  Sevilla                   |       27616 |       73.05 |    36 |   24.1 |   12.1 |      28 |    35.9
Morón de la Frontera           |  Sevilla                   |       27229 |       64.55 |   297 |   23.5 |   11.2 |      30 |    35.4
Aguadulce                      |  Sevilla                   |        2109 |      154.05 |   264 |   23.3 |   10.2 |      31 |    34.5
Lora de Estepa                 |  Sevilla               

In [3]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Provincia": "Cádiz",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Provincia':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Provincia:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Provincia                  | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------
Arcos de la Frontera           |  Cádiz                     |       30902 |       58.81 |   185 |   23.8 |   11.9 |      28 |    36.5
Trebujena                      |  Cádiz                     |        7000 |      102.45 |    69 |   23.8 |   12.6 |      27 |    36.5
Espera                         |  Cádiz                     |        3755 |       30.42 |   164 |   23.8 |   11.6 |      28 |    36.1
Jerez de la Frontera           |  Cádiz                     |      212801 |      179.19 |    56 |   23.0 |   12.8 |      25 |    38.1
Sanlúcar de Barrameda          |  Cádiz                     |       69805 |      397.23 |    30 |   22.8 |   13.4 |      23 |    37.5
Medina Sidonia                 |  Cádiz                 

In [10]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Provincia": "Huelva",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Provincia':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Provincia:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Provincia                  | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------
Rosal de la Frontera           |  Huelva                    |        1665 |         8.1 |   216 |   23.0 |   11.2 |      29 |    33.4
Huelva                         |  Huelva                    |      142538 |      958.93 |    24 |   22.8 |   13.8 |      23 |    32.9


In [2]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Provincia": "Córdoba",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Provincia':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Provincia:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Provincia                  | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------
Obejo                          |  Córdoba                   |        2101 |        9.79 |   707 |   21.8 |    9.4 |      31 |    31.8
Espiel                         |  Córdoba                   |        2326 |        5.32 |   548 |   21.1 |    8.7 |      32 |    32.2


In [16]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Provincia": "Granada",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Provincia':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Provincia:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Provincia                  | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------
Salobrena                      |  Granada                   |       12472 |      355.26 |    95 |   22.5 |   15.1 |      21 |    23.5
Motril                         |  Granada                   |       58545 |      584.84 |    45 |   22.2 |   14.9 |      21 |    22.7
Granada                        |  Granada                   |      233975 |      2658.2 |   684 |   21.9 |    7.8 |      34 |    23.6
Guadix                         |  Granada                   |       18881 |       58.23 |   913 |   19.9 |    7.2 |      32 |    18.6


In [15]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Provincia": "Almería",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Provincia':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Provincia:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Provincia                  | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------
Alicún                         |  Almería                   |         206 |       35.09 |   421 |   21.2 |   11.0 |      26 |    14.0
Níjar                          |  Almería                   |       33319 |       55.59 |   356 |   20.3 |   12.2 |      22 |    13.4


In [2]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']

pipeline = [
    # 1. Sicherstellen, dass die Felder existieren
    {
        "$match": {
            "Provincia": "Málaga",
            "Temperature high": { "$exists": True },
            "Temperature low": { "$exists": True },
            "rainfall mm": {"$exists": True}
        }
    },

    # 2. Felder in Arrays transformieren
    {
        "$project": {
            "Ciudad": 1,
            "Provincia": 1,
            "Población": 1,
            "Densidad": 1,
            "Altitud": 1,
            "high_arr": { "$objectToArray": "$Temperature high" },
            "low_arr": { "$objectToArray": "$Temperature low" },
            "rain_arr": { "$objectToArray": { "$ifNull": ["$rainfall mm", {}] } }
            
        }
    },

    # 3. Berechnungen (Durchschnitte, Min, Max)
    {
        "$addFields": {
            "avg_high": { "$avg": "$high_arr.v" },
            "avg_low": { "$avg": "$low_arr.v" },
            "max_temp": { "$max": "$high_arr.v" },
            "min_temp": { "$min": "$low_arr.v" },
            "avg_rain": { "$avg": "$rain_arr.v" },
            "water_arr": { "$objectToArray": { "$ifNull": ["$water", {}] } }
        }
    },

    # 4. Spanne und Wasser-Durchschnitt
    {
        "$addFields": {
            "avg_water": { "$avg": "$water_arr.v" },
            "amplitude": { "$subtract": ["$max_temp", "$min_temp"] }
        }
    },
    { "$sort": { "avg_high": -1 } }
]

results = collection.aggregate(pipeline)

header = f"{'Ciudad':<30} | {'Provincia':<25}  | {'Población':<10} | {'Densidad': <11} | {'Altitud':<5} |  {'High':<6} | {'Low':<6} | {'Spanne':<7} | {'Regen':<7} "
print(header)
print("-" * len(header))

for res in results:
    ciudad = res.get('Ciudad', 'Unbekannt')
    Provincia = res.get('Provincia', 'Unbekannt')
    Población = res.get('Población', 'Unbekannt')
    Densidad =  res.get('Densidad', 'Unbekannt')
    Altitud =  res.get('Altitud', 'Unbekannt')
    
    # Sicherer Fallback: Wenn ein Wert None ist, wird 0 verwendet
    a_high = round(res.get('avg_high') or 0, 1)
    a_low  = round(res.get('avg_low') or 0, 1)
    amp    = round(res.get('amplitude') or 0, 1)
    rain   = round(res.get('avg_rain') or 0, 1)
    
    w_val  = res.get('avg_water')
    water_str = f"{round(w_val, 1):>6}" if w_val is not None else "  ---  "

    print(f"{ciudad:<30} |  {Provincia:<25} |  {Población:>10} | {Densidad:>11} | {Altitud:>5} | {a_high:>6} | {a_low:>6} | {amp:>7} | {rain:>7}")

Ciudad                         | Provincia                  | Población  | Densidad    | Altitud |  High   | Low    | Spanne  | Regen   
----------------------------------------------------------------------------------------------------------------------------------------
Málaga                         |  Málaga                    |      586384 |     1440.58 |     8 |   22.8 |   13.8 |      22 |    31.6
Ronda                          |  Málaga                    |       33624 |       86.17 |   740 |   19.2 |    8.8 |      25 |    35.2
Atajate                        |  Málaga                    |         168 |       15.41 |   750 |   19.1 |    9.1 |      25 |    35.5
Pujerra                        |  Málaga                    |         285 |       11.69 |   775 |   17.9 |    9.6 |      23 |    35.4


In [11]:
from pymongo import MongoClient

# Verbindung zur MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['Espana']
collection = db['ciudad']


def check_lifestyle_fit(row):
    # Kriterium 1: Keine Touristen-Dichte (nicht zu hoch, nicht zu niedrig)
    # San Fernando (3119) wäre hier "out", Cazalla (13) wäre "in"
    density_score = 100 if 10 <= row['Densidad'] <= 500 else 0
    
    # Kriterium 2: Bahnhof-Erreichbarkeit (Karlsruher Maßstab)
    # Wir nehmen an, du hast ein Feld 'Distancia_Estacion'
    if row.get('Distancia_Estacion', 99) <= 2:
        mobility_score = 100
    elif row.get('Distancia_Estacion', 99) <= 5:
        mobility_score = 50 # E-Bike Distanz
    else:
        mobility_score = 0
        
    # Kriterium 3: Wasser-Faktor (Flussbad/Piscina Natural)
    water_score = 100 if row.get('Piscina_Natural', False) else 0
    
    total_score = (density_score + mobility_score + water_score) / 3
    return total_score

# Anwendung auf dein DataFrame
 df['Score'] = df.apply(check_lifestyle_fit, axis=1)

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 30)